# 실습 4 (심화) · 로그 파일 일괄 분석

실습 2b(마스킹)와 3(JSON 판정)을 합쳐 sample_logs.txt 12줄을 한 번에 분석하고 정답과 비교합니다.

1. 12줄 읽기 → 가명 마스킹 → 재검사
2. 한 줄씩 LLM 판정 (DoS · Exploits · Reconnaissance · Normal 중 하나)
3. 정답(`../data/sample_logs_truth.csv`)과 비교, 틀린 줄 검토

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "llm_client.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Cannot find project root containing llm_client.py")
sys.path.insert(0, str(PROJECT_ROOT))

from llm_client import create_llm_client

PROVIDER = "mono"
client, MODEL = create_llm_client(provider=PROVIDER, api_key_env="MY_API_KEY")
print(PROVIDER, MODEL)

In [ ]:
from openai.types.chat import ChatCompletionMessageParam

def ask(
    prompt: str,
    system: str | None = None,
    temperature: float = 0.2,
    show: bool = True,
) -> str:
    """질문을 보내고 답을 받아 돌려줌. show=True면 답이 만들어지는 대로 화면에 표시"""
    messages: list[ChatCompletionMessageParam] = [
        {"role": "user", "content": prompt}
    ]
    if system:
        messages.insert(0, {"role": "system", "content": system})
    if PROVIDER == "ollama":
        stream = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            temperature=temperature,
            stream=True,
            max_tokens=500,
        )
    else:
        stream = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            temperature=temperature,
            stream=True,
        )
    answer = ""
    for chunk in stream:
        piece = chunk.choices[0].delta.content if chunk.choices else None
        if piece:
            answer += piece
            if show:
                print(piece, end="", flush=True)
    if show:
        print()
    return answer

In [4]:
import json, re

def parse_json(raw):
    """답에서 { ... } 부분만 꺼내 JSON으로 읽음. 없거나 깨졌으면 None"""
    m = re.search(r"\{.*\}", raw or "", re.DOTALL)
    try:
        return json.loads(m.group(0)) if m else None
    except json.JSONDecodeError:
        return None

## 1) 읽기 · 가명 마스킹 · 재검사 (실습 2b와 같은 규칙)

In [5]:
import re

INTERNAL_IP = r"\b149\.171\.126\.\d{1,3}\b"     # 우리 서버 대역

def mask_logs(lines):
    table = {}
    for line in lines:
        for ip in re.findall(INTERNAL_IP, line):
            table.setdefault(ip, f"SRV_{len(table) + 1:03d}")
    return [re.sub(INTERNAL_IP, lambda m: table[m.group(0)], l) for l in lines], table

LINES = open("sample_logs.txt", encoding="utf-8").read().splitlines()
masked, table = mask_logs(LINES)
assert not any(re.search(INTERNAL_IP, l) for l in masked), "마스킹 누락 → 전송 중단"
print(f"{len(masked)}줄, 내부 서버 {len(table)}개 → SRV_001~ (가명표는 PC 안에만)")

12줄, 내부 서버 8개 → SRV_001~ (가명표는 PC 안에만)


## 2) 한 줄씩 판정

In [6]:
TYPES = ["DoS", "Exploits", "Reconnaissance", "Normal"]
PROMPT = """한 줄 = 네트워크 연결 1건 (UNSW-NB15). src/dst=출발지/목적지 IP:포트, state=연결 상태(FIN 정상 종료, INT 응답 없이 끝남),
spkts/dpkts=송신/수신 패킷, sbytes/dbytes=송신/수신 바이트, rate=초당 패킷 수. SRV_001 같은 값은 가명 처리한 우리 서버.
공격 유형은 DoS(서비스 방해) · Exploits(취약점 침투) · Reconnaissance(포트 · 서비스 탐색) · Normal(정상) 중 하나만 고른다.

로그: {log}

JSON으로만 답한다.  
{{"attack_type": "DoS / Exploits / Reconnaissance / Normal 중 하나", "risk_score": 0~100 정수, "reason": "한 줄 설명"}}"""

def analyze(log):
    for _ in range(2):
        d = parse_json(ask(PROMPT.format(log=log), system="JSON만 출력하는 보안 분석가.", show=False))
        if d and d.get("attack_type") in TYPES:
            return d
    return {"attack_type": "판정불가", "risk_score": None, "reason": "형식 오류 → 사람 검토"}

results = []
for i, line in enumerate(masked, 1):
    print(f"[{i}/{len(masked)}] 분석 중 ...", end=" ", flush=True)
    r = analyze(line[28:])
    print(r["attack_type"], "|", str(r.get("reason"))[:70])
    results.append(dict(r, line=i))

[1/12] 분석 중 ... Exploits | 알려진 외부 공격자 대역으로 보이는 src가 HTTP(80) 서버와 정상 세션(FIN)을 맺고 데이터를 교환해 단순 스캔/Do
[2/12] 분석 중 ... Normal | 정상 종료(FIN)된 낮은 속도의 일반적인 HTTP 연결이며 패킷·바이트 수와 목적지 접근 빈도에 이상 징후가 적음
[3/12] 분석 중 ... Normal | DNS 질의/응답이 정상적으로 양방향 2패킷씩 완료되었고 패킷·바이트 수와 연결 카운트가 낮아 이상 징후가 거의 없습니다.
[4/12] 분석 중 ... Normal | UDP 111 포트로 짧은 양방향 응답(CON)이 있고 패킷·바이트 수와 연결 빈도가 낮아 탐색이나 DoS 징후가 약함
[5/12] 분석 중 ... Normal | FIN 정상 종료된 짧은 ftp-data 세션이며 패킷·바이트 규모와 연결 빈도가 낮아 공격 징후가 뚜렷하지 않음
[6/12] 분석 중 ... Normal | SMTP 연결이 FIN으로 정상 종료되었고 양방향 패킷/바이트 교환과 낮은 rate로 볼 때 공격 징후가 약함
[7/12] 분석 중 ... Normal | SSH 연결이 FIN으로 정상 종료되었고 송수신 패킷/바이트가 균형적이며 탐색·공격 징후가 낮습니다.
[8/12] 분석 중 ... Reconnaissance | UDP 111번 포트로 응답 없이 짧은 패킷만 전송되어 서비스 존재 여부를 확인하는 포트/서비스 탐색 패턴에 가깝습니다.
[9/12] 분석 중 ... Exploits | 외부 공격원으로 알려진 대역에서 RPC 포트 111로 TCP 세션을 정상 종료하며 소량의 양방향 트래픽을 주고받아 서비스 취약
[10/12] 분석 중 ... Exploits | 알려진 공격자 대역으로 보이는 외부 출발지가 내부 HTTP 서버(80/tcp)에 정상 종료된 단일 요청을 수행했으며, 스캔·D
[11/12] 분석 중 ... Exploits | HTTP 80번으로 정상 종료된 단일 세션이지만 외부 출발지에서 서버로 요청

## 3) 정답과 비교

In [12]:
import pandas as pd

truth = pd.read_csv("../data/sample_logs_truth.csv")
table = pd.DataFrame(results).merge(truth[["line", "attack_cat"]], on="line")
table["일치"] = table["attack_type"] == table["attack_cat"]
print(table[["line", "attack_type", "attack_cat", "risk_score", "일치"]].to_string(index=False))
print(f"\n일치 {table['일치'].sum()}/{len(table)}, 판정불가 {(table['attack_type'] == '판정불가').sum()}건")
print("\n틀린 줄의 이유:")
for _, r in table[~table["일치"]].iterrows():                   #일치하지 않은 로그만 
    print(f"  {r['line']}번째 (정답 {r['attack_cat']}, 예측 {r['attack_type']}): {str(r['reason'])[:90]}")

 line    attack_type     attack_cat  risk_score    일치
    1         Normal            DoS          15 False
    2         Normal         Normal           8  True
    3         Normal         Normal           5  True
    4         Normal         Normal          15  True
    5         Normal         Normal          12  True
    6         Normal         Normal          12  True
    7         Normal         Normal           5  True
    8 Reconnaissance Reconnaissance          55  True
    9 Reconnaissance Reconnaissance          55  True
   10       Exploits       Exploits          65  True
   11         Normal            DoS          15 False
   12       Exploits       Exploits          78  True

일치 10/12, 판정불가 0건

틀린 줄의 이유:
  1번째 (정답 DoS, 예측 Normal): HTTP 연결이 FIN으로 정상 종료되었고 패킷·바이트 수와 단일 목적지/포트 카운트가 낮아 단일 로그상 공격 징후가 약함
  11번째 (정답 DoS, 예측 Normal): HTTP 연결이 FIN으로 정상 종료되었고 패킷·바이트 수와 동일 목적지/포트 반복 횟수가 낮아 DoS·침투·정찰 징후가 약함


### 토론
- 1 · 10 · 11번째 줄처럼 정상 웹 연결과 크기가 같은 공격을 LLM이 어떻게 판정했는지 봅니다.
- 같은 출발지(175.45.176.1)의 8~12번째 줄은 여러 줄을 함께 봐야 정찰 → 침투 흐름이 보입니다. 한 줄씩 분석하면 잡히는지 토론합니다.